In [1]:
!pip install -q faster-whisper
!pip install -q "transformers>=4.44.0" accelerate
!pip install -q sentencepiece sacremoses
!pip install -q python-docx scipy
!apt-get install -q ffmpeg
!pip install -q gtts
print("✅ Done — restart runtime now")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.7 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 w

In [1]:
import os, re, json, torch, warnings
from datetime import datetime
from collections import defaultdict
from faster_whisper import WhisperModel
from transformers import (
    BartForConditionalGeneration, BartTokenizer,
    MBartForConditionalGeneration, MBart50TokenizerFast,
    pipeline
)
from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.oxml import OxmlElement
from google.colab import files

warnings.filterwarnings('ignore')
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE   = "float16" if DEVICE == "cuda" else "int8"
HF_DEVICE = 0 if DEVICE == "cuda" else -1
print(f"✅ Running on : {DEVICE.upper()}")

✅ Running on : CPU


In [9]:
# ── CONFIGURE YOUR LANGUAGES HERE ─────────────
SOURCE_LANG = "auto"   # "auto" = Whisper detects it automatically
                       # OR set manually e.g. "hi" "fr" "de" "ar" "te" "ta"

TARGET_LANG = "hi"     # Language you want the output translated TO
                       # Examples:
                       # "fr" = French    "de" = German
                       # "hi" = Hindi     "ar" = Arabic
                       # "es" = Spanish   "zh" = Chinese
                       # "ja" = Japanese  "ru" = Russian
                       # "te" = Telugu    "ta" = Tamil
                       # "ko" = Korean    "pt" = Portuguese
                       # "it" = Italian   "nl" = Dutch
                       # "tr" = Turkish   "uk" = Ukrainian
# ──────────────────────────────────────────────

# Full language name map for display
LANG_NAMES = {
    "en":"English","fr":"French","de":"German","hi":"Hindi",
    "ar":"Arabic","es":"Spanish","zh":"Chinese","ja":"Japanese",
    "ru":"Russian","te":"Telugu","ta":"Tamil","ko":"Korean",
    "pt":"Portuguese","it":"Italian","nl":"Dutch","tr":"Turkish",
    "uk":"Ukrainian","pl":"Polish","sv":"Swedish","fi":"Finnish",
    "ro":"Romanian","vi":"Vietnamese","th":"Thai","id":"Indonesian",
    "ur":"Urdu","bn":"Bengali","gu":"Gujarati","mr":"Marathi"
}

print(f"🌐 Target Language : {LANG_NAMES.get(TARGET_LANG, TARGET_LANG)}")
print(f"🎙️  Source Language : {'Auto-detect' if SOURCE_LANG == 'auto' else LANG_NAMES.get(SOURCE_LANG, SOURCE_LANG)}")

🌐 Target Language : Hindi
🎙️  Source Language : Auto-detect


In [3]:
from gtts import gTTS
import IPython.display as ipd

# Change lang= to test different source languages
# "en"=English "hi"=Hindi "fr"=French "de"=German "es"=Spanish
TEXT_LANG = "en"

text = """
Good morning everyone. Let's start the Q3 project review.
John, can you make sure the unit tests are complete by Thursday?
Absolutely, I will have all tests ready by end of day Thursday.
We have decided to move the launch date to October 8th.
Sarah, please send the client a confirmation email today.
We also agreed that the mobile team will handle the iOS bug fixes.
Mike, please schedule a technical review with the database team by Monday.
Sure, I will set that up and share the calendar invite.
Next meeting is Wednesday at 10 AM. Thank you everyone.
"""

tts = gTTS(text=text.strip(), lang=TEXT_LANG, slow=False)
tts.save("meeting.mp3")
AUDIO_PATH = "meeting.mp3"
print("✅ Test audio ready")
ipd.display(ipd.Audio("meeting.mp3"))

✅ Test audio ready


In [4]:
print("Loading Whisper model...")
whisper_model = WhisperModel("base", device=DEVICE, compute_type=COMPUTE)

print("Transcribing and detecting language...")
raw_segments, info = whisper_model.transcribe(
    AUDIO_PATH,
    beam_size=5,
    language=None if SOURCE_LANG == "auto" else SOURCE_LANG
)

segments = []
for seg in raw_segments:
    segments.append({
        "start" : round(seg.start, 2),
        "end"   : round(seg.end,   2),
        "text"  : seg.text.strip()
    })

transcript       = " ".join(s["text"] for s in segments)
detected_lang    = info.language
detected_prob    = round(info.language_probability * 100, 1)

print(f"\n✅ Transcription done!")
print(f"   Detected Language : {LANG_NAMES.get(detected_lang, detected_lang)} ({detected_prob}% confidence)")
print(f"   Segments          : {len(segments)}")
print(f"   Words             : {len(transcript.split())}")
print(f"\nPreview: {transcript[:300]}")

Loading Whisper model...


Transcribing and detecting language...

✅ Transcription done!
   Detected Language : English (99.5% confidence)
   Segments          : 6
   Words             : 99

Preview: Good morning everyone. Let's start the Q3 project review. John, can you make sure the unit tests are complete by Thursday? Absolutely. I will have all tests ready by end of day Thursday. We have decided to move the launch date to October 8. Sarah, please send the client a confirmation email today. W


In [5]:
def simple_diarize(segments, num_speakers=3):
    out, spk, last_end = [], 1, 0.0
    for seg in segments:
        if seg["start"] - last_end >= 1.5:
            spk = (spk % num_speakers) + 1
        out.append({
            "speaker" : f"Speaker {spk}",
            "start"   : seg["start"],
            "end"     : seg["end"],
            "text"    : seg["text"]
        })
        last_end = seg["end"]
    return out

diarized = simple_diarize(segments, num_speakers=3)
print(f"✅ {len(diarized)} segments diarized")
for s in diarized[:4]:
    print(f"  [{s['start']}s] {s['speaker']}: {s['text'][:60]}")

✅ 6 segments diarized
  [0.0s] Speaker 1: Good morning everyone. Let's start the Q3 project review. Jo
  [7.04s] Speaker 1: are complete by Thursday? Absolutely. I will have all tests 
  [13.92s] Speaker 1: We have decided to move the launch date to October 8. Sarah,
  [19.84s] Speaker 1: confirmation email today. We also agreed that the mobile tea


In [6]:
# ── Summarizer ────────────────────────────────
print("Loading summarizer...")
sum_tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
sum_model     = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn").to(DEVICE)
print("✅ Summarizer ready")

# ── Zero-shot Classifier ──────────────────────
print("Loading classifier...")
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli",
                      device=HF_DEVICE)
print("✅ Classifier ready")

# ── Translation Model (mBART 50 languages) ────
print("Loading translation model (facebook/mbart-large-50-many-to-many-mmt)...")
trans_tokenizer = MBart50TokenizerFast.from_pretrained(
    "facebook/mbart-large-50-many-to-many-mmt"
)
trans_model = MBartForConditionalGeneration.from_pretrained(
    "facebook/mbart-large-50-many-to-many-mmt"
).to(DEVICE)
print("✅ Translation model ready")

# ── mBART language code map ───────────────────
MBART_LANG_MAP = {
    "ar":"ar_AR","cs":"cs_CZ","de":"de_DE","en":"en_XX",
    "es":"es_XX","et":"et_EE","fi":"fi_FI","fr":"fr_XX",
    "gu":"gu_IN","hi":"hi_IN","it":"it_IT","ja":"ja_XX",
    "kk":"kk_KZ","ko":"ko_KR","lt":"lt_LT","lv":"lv_LV",
    "my":"my_MM","ne":"ne_NP","nl":"nl_XX","ro":"ro_RO",
    "ru":"ru_RU","si":"si_LK","tr":"tr_TR","vi":"vi_VN",
    "zh":"zh_CN","af":"af_ZA","az":"az_AZ","bn":"bn_IN",
    "fa":"fa_IR","he":"he_IL","hr":"hr_HR","id":"id_ID",
    "ka":"ka_GE","km":"km_KH","mk":"mk_MK","ml":"ml_IN",
    "mn":"mn_MN","mr":"mr_IN","pl":"pl_PL","ps":"ps_AF",
    "pt":"pt_XX","sv":"sv_SE","sw":"sw_KE","ta":"ta_IN",
    "te":"te_IN","th":"th_TH","tl":"tl_XX","uk":"uk_UA",
    "ur":"ur_PK","xh":"xh_ZA","gl":"gl_ES","sl":"sl_SI"
}
print(f"\n🌐 Supported languages: {len(MBART_LANG_MAP)}")

Loading summarizer...


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

✅ Translation model ready

🌐 Supported languages: 52


In [7]:
def get_summary(text):
    words  = text.split()
    chunks = [" ".join(words[i:i+700]) for i in range(0, len(words), 700)]
    parts  = []
    for c in chunks:
        if len(c.split()) > 30:
            inputs = sum_tokenizer(c, return_tensors="pt",
                                   max_length=1024, truncation=True)
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            ids    = sum_model.generate(inputs["input_ids"],
                                        max_length=130, min_length=30,
                                        length_penalty=2.0, num_beams=4)
            parts.append(sum_tokenizer.decode(ids[0], skip_special_tokens=True))
    return " ".join(parts)

def translate_text(text, src_lang, tgt_lang):
    """Translate text from src_lang to tgt_lang using mBART"""
    if src_lang == tgt_lang:
        return text
    src_code = MBART_LANG_MAP.get(src_lang, "en_XX")
    tgt_code = MBART_LANG_MAP.get(tgt_lang, "fr_XX")

    trans_tokenizer.src_lang = src_code
    inputs = trans_tokenizer(
        text, return_tensors="pt",
        max_length=512, truncation=True, padding=True
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    tgt_id = trans_tokenizer.lang_code_to_id[tgt_code]

    out = trans_model.generate(
        **inputs,
        forced_bos_token_id=tgt_id,
        max_length=512,
        num_beams=4
    )
    return trans_tokenizer.decode(out[0], skip_special_tokens=True)

def get_actions(segs):
    kws    = ["will","should","need to","must","please","send",
              "schedule","review","ensure","prepare","follow up"]
    labels = ["action item","task assignment","general discussion"]
    out    = []
    for s in segs:
        if any(k in s["text"].lower() for k in kws):
            r = classifier(s["text"], candidate_labels=labels)
            if r["labels"][0] != "general discussion" and r["scores"][0] > 0.40:
                out.append({"speaker": s["speaker"],
                            "text"   : s["text"],
                            "time"   : f"{s['start']}s"})
    return out

def get_decisions(segs):
    kws    = ["decided","agreed","approved","confirmed","moving forward",
              "we will","finalized","going with","accepted"]
    labels = ["decision made","agreement reached","general statement"]
    out    = []
    for s in segs:
        if any(k in s["text"].lower() for k in kws):
            r = classifier(s["text"], candidate_labels=labels)
            if r["labels"][0] != "general statement" and r["scores"][0] > 0.38:
                out.append({"speaker": s["speaker"],
                            "text"   : s["text"],
                            "time"   : f"{s['start']}s"})
    return out

# ── Run extraction ────────────────────────────
print("Extracting summary...")
summary = get_summary(transcript)

print("Extracting actions...")
actions = get_actions(diarized)

print("Extracting decisions...")
decisions = get_decisions(diarized)

# ── Translate everything to target language ───
tgt_name = LANG_NAMES.get(TARGET_LANG, TARGET_LANG)
src_name = LANG_NAMES.get(detected_lang, detected_lang)
print(f"\nTranslating {src_name} → {tgt_name}...")

summary_translated = translate_text(summary, detected_lang, TARGET_LANG)

for a in actions:
    a["text_translated"] = translate_text(a["text"], detected_lang, TARGET_LANG)

for d in decisions:
    d["text_translated"] = translate_text(d["text"], detected_lang, TARGET_LANG)

for s in diarized:
    s["text_translated"] = translate_text(s["text"], detected_lang, TARGET_LANG)

print(f"\n✅ Summary     : {len(summary.split())} words")
print(f"✅ Actions     : {len(actions)}")
print(f"✅ Decisions   : {len(decisions)}")
print(f"\nORIGINAL SUMMARY:\n{summary}")
print(f"\nTRANSLATED SUMMARY ({tgt_name}):\n{summary_translated}")

Extracting summary...
Extracting actions...
Extracting decisions...

Translating English → French...

✅ Summary     : 31 words
✅ Actions     : 5
✅ Decisions   : 2

ORIGINAL SUMMARY:
We have decided to move the launch date to October 8. We also agreed that the mobile team will handle the iOS bug fixes. Next meeting is Wednesday at 10 a.m.

TRANSLATED SUMMARY (French):
Nous avons décidé de reporter la date de lancement au 8 octobre et nous avons également convenu que l'équipe mobile s'occupera des corrections de bugs iOS.


In [8]:
def set_cell_bg(cell, hex_color):
    tc   = cell._tc
    tcPr = tc.get_or_add_tcPr()
    shd  = OxmlElement("w:shd")
    shd.set(qn("w:val"),   "clear")
    shd.set(qn("w:color"), "auto")
    shd.set(qn("w:fill"),  hex_color)
    tcPr.append(shd)

def add_banner(doc, title, color="1F497D"):
    t = doc.add_table(rows=1, cols=1)
    c = t.cell(0, 0)
    set_cell_bg(c, color)
    p = c.paragraphs[0]
    r = p.add_run(f"  {title}")
    r.bold = True
    r.font.size = Pt(11)
    r.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
    doc.add_paragraph()

def add_bilingual_item(doc, original, translated, speaker, time, orig_color, tgt_name):
    p = doc.add_paragraph(style="List Number")
    sr = p.add_run(f"[{speaker}]  ")
    sr.bold = True
    sr.font.color.rgb = orig_color
    p.add_run(original)
    p.add_run(f"  ({time})").italic = True

    # Translated line below in grey
    tp = doc.add_paragraph()
    tp.paragraph_format.left_indent = Inches(0.4)
    tr_ = tp.add_run(f"  🌐 {tgt_name}: ")
    tr_.bold = True
    tr_.font.size = Pt(9)
    tr_.font.color.rgb = RGBColor(0x70, 0x70, 0x70)
    tt = tp.add_run(translated)
    tt.font.size = Pt(9)
    tt.font.color.rgb = RGBColor(0x50, 0x50, 0x50)
    tt.italic = True

def export_bilingual_docx(summary, summary_translated,
                           actions, decisions, diarized,
                           src_lang, tgt_lang):
    doc  = Document()
    sec  = doc.sections[0]
    sec.left_margin = sec.right_margin = Inches(1.0)
    sec.top_margin  = sec.bottom_margin = Inches(0.8)

    src_name = LANG_NAMES.get(src_lang, src_lang)
    tgt_name = LANG_NAMES.get(tgt_lang, tgt_lang)

    # ── Title ─────────────────────────────────
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = p.add_run("MINUTES OF MEETING")
    r.bold = True
    r.font.size = Pt(16)
    r.font.color.rgb = RGBColor(0x1F, 0x49, 0x7D)

    sub = doc.add_paragraph()
    sub.alignment = WD_ALIGN_PARAGRAPH.CENTER
    sub.add_run(f"Original: {src_name}   |   Translated: {tgt_name}").italic = True

    doc.add_paragraph(datetime.now().strftime("%B %d, %Y  |  %I:%M %p"))
    doc.add_paragraph()

    # ── Summary ───────────────────────────────
    add_banner(doc, "1.  Executive Summary", "1F497D")
    doc.add_paragraph(f"[{src_name}]").runs[0].bold = True
    doc.add_paragraph(summary)
    doc.add_paragraph(f"[{tgt_name}]").runs[0].bold = True
    p2 = doc.add_paragraph(summary_translated)
    p2.runs[0].italic = True
    doc.add_paragraph()

    # ── Decisions ─────────────────────────────
    add_banner(doc, "2.  Key Decisions", "375623")
    if decisions:
        for d in decisions:
            add_bilingual_item(
                doc,
                d["text"], d["text_translated"],
                d["speaker"], d["time"],
                RGBColor(0x37, 0x56, 0x23), tgt_name
            )
    else:
        doc.add_paragraph("No decisions extracted.")
    doc.add_paragraph()

    # ── Actions ───────────────────────────────
    add_banner(doc, "3.  Action Items", "C00000")
    if actions:
        for a in actions:
            add_bilingual_item(
                doc,
                a["text"], a["text_translated"],
                a["speaker"], a["time"],
                RGBColor(0xC0, 0x00, 0x00), tgt_name
            )
    else:
        doc.add_paragraph("No action items extracted.")
    doc.add_paragraph()

    # ── Bilingual Transcript ──────────────────
    add_banner(doc, f"4.  Full Transcript  ({src_name} + {tgt_name})", "595959")
    for s in diarized:
        p = doc.add_paragraph()
        r = p.add_run(f"[{s['start']}s]  {s['speaker']}: ")
        r.bold = True; r.font.size = Pt(9)
        p.add_run(s["text"]).font.size = Pt(9)

        tp = doc.add_paragraph()
        tp.paragraph_format.left_indent = Inches(0.4)
        tr_ = tp.add_run(f"  🌐 {tgt_name}: ")
        tr_.font.size = Pt(8)
        tr_.font.color.rgb = RGBColor(0x70, 0x70, 0x70)
        tt = tp.add_run(s["text_translated"])
        tt.font.size = Pt(8)
        tt.italic = True
        tt.font.color.rgb = RGBColor(0x50, 0x50, 0x50)

    path = f"Meeting_Minutes_{src_name}_to_{tgt_name}.docx"
    doc.save(path)
    print(f"✅ Saved: {path}")
    return path

path = export_bilingual_docx(
    summary, summary_translated,
    actions, decisions, diarized,
    detected_lang, TARGET_LANG
)
files.download(path)
print("📥 Check your downloads folder!")

✅ Saved: Meeting_Minutes_English_to_French.docx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Check your downloads folder!


In [1]:
!pip install -q streamlit pyngrok
print("✅ Done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 120.7 MB/s eta 0:00:00
✅ Done


In [6]:
%%writefile app.py
import os
import torch
import warnings
import streamlit as st
from datetime import datetime
from faster_whisper import WhisperModel
from transformers import (
    BartForConditionalGeneration, BartTokenizer,
    MBartForConditionalGeneration, MBart50TokenizerFast,
    pipeline
)
from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

warnings.filterwarnings("ignore")

DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE   = "float16" if DEVICE == "cuda" else "int8"
HF_DEVICE = 0 if DEVICE == "cuda" else -1

LANG_NAMES = {
    "en":"English","fr":"French","de":"German","hi":"Hindi",
    "ar":"Arabic","es":"Spanish","zh":"Chinese","ja":"Japanese",
    "ru":"Russian","te":"Telugu","ta":"Tamil","ko":"Korean",
    "pt":"Portuguese","it":"Italian","nl":"Dutch","tr":"Turkish",
    "uk":"Ukrainian","pl":"Polish","sv":"Swedish","fi":"Finnish"
}

MBART_LANG_MAP = {
    "ar":"ar_AR","de":"de_DE","en":"en_XX","es":"es_XX",
    "fi":"fi_FI","fr":"fr_XX","hi":"hi_IN","id":"id_ID",
    "it":"it_IT","ja":"ja_XX","ko":"ko_KR","nl":"nl_XX",
    "pl":"pl_PL","pt":"pt_XX","ro":"ro_RO","ru":"ru_RU",
    "sv":"sv_SE","ta":"ta_IN","te":"te_IN","th":"th_TH",
    "tr":"tr_TR","uk":"uk_UA","vi":"vi_VN","zh":"zh_CN"
}

@st.cache_resource
def load_whisper():
    return WhisperModel("base", device=DEVICE, compute_type=COMPUTE)

@st.cache_resource
def load_summarizer():
    tok = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
    mdl = BartForConditionalGeneration.from_pretrained(
              "facebook/bart-large-cnn").to(DEVICE)
    return tok, mdl

@st.cache_resource
def load_classifier():
    return pipeline("zero-shot-classification",
                    model="facebook/bart-large-mnli",
                    device=HF_DEVICE)

@st.cache_resource
def load_translator():
    tok = MBart50TokenizerFast.from_pretrained(
              "facebook/mbart-large-50-many-to-many-mmt")
    mdl = MBartForConditionalGeneration.from_pretrained(
              "facebook/mbart-large-50-many-to-many-mmt").to(DEVICE)
    return tok, mdl

def transcribe(audio_path):
    model = load_whisper()
    raw, info = model.transcribe(audio_path, beam_size=5, language=None)
    segments = [{
        "start": round(s.start, 2),
        "end"  : round(s.end,   2),
        "text" : s.text.strip()
    } for s in raw]
    transcript = " ".join(s["text"] for s in segments)
    return transcript, segments, info.language, round(info.language_probability*100, 1)

def simple_diarize(segments, num_speakers=3):
    out, spk, last_end = [], 1, 0.0
    for seg in segments:
        if seg["start"] - last_end >= 1.5:
            spk = (spk % num_speakers) + 1
        out.append({**seg, "speaker": f"Speaker {spk}"})
        last_end = seg["end"]
    return out

def get_summary(text):
    tok, mdl = load_summarizer()
    words  = text.split()
    chunks = [" ".join(words[i:i+700]) for i in range(0, len(words), 700)]
    parts  = []
    for c in chunks:
        if len(c.split()) > 30:
            inp = tok(c, return_tensors="pt", max_length=1024, truncation=True)
            inp = {k: v.to(DEVICE) for k, v in inp.items()}
            ids = mdl.generate(inp["input_ids"], max_length=130,
                               min_length=30, num_beams=4)
            parts.append(tok.decode(ids[0], skip_special_tokens=True))
    return " ".join(parts)

def get_actions(segs):
    clf  = load_classifier()
    kws  = ["will","should","need to","must","please","send",
            "schedule","review","ensure","prepare","follow up"]
    lbls = ["action item","task assignment","general discussion"]
    out  = []
    for s in segs:
        if any(k in s["text"].lower() for k in kws):
            r = clf(s["text"], candidate_labels=lbls)
            if r["labels"][0] != "general discussion" and r["scores"][0] > 0.40:
                out.append({
                    "speaker": s["speaker"],
                    "text"   : s["text"],
                    "time"   : str(s["start"]) + "s"
                })
    return out

def get_decisions(segs):
    clf  = load_classifier()
    kws  = ["decided","agreed","approved","confirmed","moving forward",
            "we will","finalized","going with","accepted"]
    lbls = ["decision made","agreement reached","general statement"]
    out  = []
    for s in segs:
        if any(k in s["text"].lower() for k in kws):
            r = clf(s["text"], candidate_labels=lbls)
            if r["labels"][0] != "general statement" and r["scores"][0] > 0.38:
                out.append({
                    "speaker": s["speaker"],
                    "text"   : s["text"],
                    "time"   : str(s["start"]) + "s"
                })
    return out

def translate_text(text, src, tgt):
    if src == tgt or tgt not in MBART_LANG_MAP:
        return text
    tok, mdl = load_translator()
    tok.src_lang = MBART_LANG_MAP.get(src, "en_XX")
    inp = tok(text, return_tensors="pt", max_length=512,
              truncation=True, padding=True)
    inp = {k: v.to(DEVICE) for k, v in inp.items()}
    tgt_id = tok.lang_code_to_id[MBART_LANG_MAP[tgt]]
    out = mdl.generate(**inp, forced_bos_token_id=tgt_id,
                       max_length=512, num_beams=4)
    return tok.decode(out[0], skip_special_tokens=True)

def set_cell_bg(cell, hex_color):
    tc   = cell._tc
    tcPr = tc.get_or_add_tcPr()
    shd  = OxmlElement("w:shd")
    shd.set(qn("w:val"),   "clear")
    shd.set(qn("w:color"), "auto")
    shd.set(qn("w:fill"),  hex_color)
    tcPr.append(shd)

def add_banner(doc, title, color="1F497D"):
    t = doc.add_table(rows=1, cols=1)
    c = t.cell(0, 0)
    set_cell_bg(c, color)
    p = c.paragraphs[0]
    r = p.add_run("  " + title)
    r.bold = True
    r.font.size = Pt(11)
    r.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
    doc.add_paragraph()

def export_docx(summary, summary_tr, actions, decisions,
                diarized, src_lang, tgt_lang):
    doc = Document()
    sec = doc.sections[0]
    sec.left_margin = sec.right_margin = Inches(1.0)
    sec.top_margin  = sec.bottom_margin = Inches(0.8)

    src_name = LANG_NAMES.get(src_lang, src_lang)
    tgt_name = LANG_NAMES.get(tgt_lang, tgt_lang)

    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = p.add_run("MINUTES OF MEETING")
    r.bold = True
    r.font.size = Pt(16)
    r.font.color.rgb = RGBColor(0x1F, 0x49, 0x7D)
    s = doc.add_paragraph()
    s.alignment = WD_ALIGN_PARAGRAPH.CENTER
    s.add_run("Original: " + src_name + "   |   Translated: " + tgt_name).italic = True
    doc.add_paragraph(datetime.now().strftime("%B %d, %Y  |  %I:%M %p"))
    doc.add_paragraph()

    add_banner(doc, "1.  Executive Summary", "1F497D")
    doc.add_paragraph(summary)
    p2 = doc.add_paragraph(summary_tr)
    p2.runs[0].italic = True
    doc.add_paragraph()

    add_banner(doc, "2.  Key Decisions", "375623")
    if decisions:
        for d in decisions:
            p = doc.add_paragraph(style="List Number")
            r = p.add_run("[" + d["speaker"] + "]  ")
            r.bold = True
            r.font.color.rgb = RGBColor(0x37, 0x56, 0x23)
            p.add_run(d["text"])
            p.add_run("  (" + d["time"] + ")").italic = True
            tp = doc.add_paragraph()
            tp.paragraph_format.left_indent = Inches(0.4)
            tr_run = tp.add_run("  🌐 " + tgt_name + ": ")
            tr_run.font.size = Pt(9)
            tt = tp.add_run(d.get("text_tr", ""))
            tt.font.size = Pt(9)
            tt.italic = True
    else:
        doc.add_paragraph("No decisions extracted.")
    doc.add_paragraph()

    add_banner(doc, "3.  Action Items", "C00000")
    if actions:
        for a in actions:
            p = doc.add_paragraph(style="List Number")
            r = p.add_run("[" + a["speaker"] + "]  ")
            r.bold = True
            r.font.color.rgb = RGBColor(0xC0, 0x00, 0x00)
            p.add_run(a["text"])
            p.add_run("  (" + a["time"] + ")").italic = True
            tp = doc.add_paragraph()
            tp.paragraph_format.left_indent = Inches(0.4)
            tr_run = tp.add_run("  🌐 " + tgt_name + ": ")
            tr_run.font.size = Pt(9)
            tt = tp.add_run(a.get("text_tr", ""))
            tt.font.size = Pt(9)
            tt.italic = True
    else:
        doc.add_paragraph("No action items extracted.")
    doc.add_paragraph()

    add_banner(doc, "4.  Full Transcript (" + src_name + " + " + tgt_name + ")", "595959")
    for s in diarized:
        p = doc.add_paragraph()
        r = p.add_run("[" + str(s["start"]) + "s]  " + s["speaker"] + ": ")
        r.bold = True
        r.font.size = Pt(9)
        p.add_run(s["text"]).font.size = Pt(9)
        tp = doc.add_paragraph()
        tp.paragraph_format.left_indent = Inches(0.4)
        tr_run = tp.add_run("  🌐 " + tgt_name + ": ")
        tr_run.font.size = Pt(8)
        tt = tp.add_run(s.get("text_tr", ""))
        tt.font.size = Pt(8)
        tt.italic = True

    path = "Meeting_Minutes_" + src_name + "_to_" + tgt_name + ".docx"
    doc.save(path)
    return path


# ══════════════════════════════════════════════
#  STREAMLIT UI
# ══════════════════════════════════════════════
st.set_page_config(
    page_title="MoM Automation System",
    page_icon="📋",
    layout="wide"
)

st.markdown("""
    <h1 style='text-align:center; color:#1F497D;'>
        📋 Minutes of Meeting Automation
    </h1>
    <p style='text-align:center; color:gray;'>
        Upload any meeting audio → Get bilingual formatted minutes
    </p>
    <hr>
""", unsafe_allow_html=True)

with st.sidebar:
    st.header("⚙️ Settings")
    target_lang = st.selectbox(
        "Translate output to:",
        options=list(LANG_NAMES.keys()),
        format_func=lambda x: LANG_NAMES[x] + " (" + x + ")",
        index=list(LANG_NAMES.keys()).index("hi")
    )
    num_speakers = st.slider(
        "Estimated number of speakers",
        min_value=1, max_value=6, value=3
    )
    st.markdown("---")
    st.markdown("**Supported formats:** `.mp3` `.wav` `.m4a` `.mp4`")
    st.markdown("---")
    st.markdown("**Models**")
    st.markdown("- faster-whisper")
    st.markdown("- BART-large-CNN")
    st.markdown("- BART-large-MNLI")
    st.markdown("- mBART-50")

uploaded = st.file_uploader(
    "📁 Upload your meeting audio",
    type=["mp3","wav","m4a","mp4"]
)

if uploaded:
    audio_path = "uploaded_" + uploaded.name
    with open(audio_path, "wb") as f:
        f.write(uploaded.read())

    st.audio(uploaded)
    st.success("✅ Uploaded: " + uploaded.name)

    if st.button("🚀 Generate Meeting Minutes", type="primary"):

        with st.spinner("🎙️ Transcribing audio..."):
            transcript, segments, det_lang, det_prob = transcribe(audio_path)

        st.info("🌐 Detected: **" + LANG_NAMES.get(det_lang, det_lang) + "** (" + str(det_prob) + "% confidence)")

        with st.spinner("👥 Assigning speakers..."):
            diarized = simple_diarize(segments, num_speakers)

        with st.spinner("🧠 Extracting summary, actions, decisions..."):
            summary   = get_summary(transcript)
            actions   = get_actions(diarized)
            decisions = get_decisions(diarized)

        tgt_name = LANG_NAMES.get(target_lang, target_lang)
        with st.spinner("🌐 Translating to " + tgt_name + "..."):
            summary_tr = translate_text(summary, det_lang, target_lang)
            for a in actions:
                a["text_tr"] = translate_text(a["text"], det_lang, target_lang)
            for d in decisions:
                d["text_tr"] = translate_text(d["text"], det_lang, target_lang)
            for s in diarized:
                s["text_tr"] = translate_text(s["text"], det_lang, target_lang)

        st.success("✅ Processing complete!")
        st.markdown("---")

        col1, col2, col3 = st.columns(3)
        col1.metric("Action Items",        len(actions))
        col2.metric("Key Decisions",       len(decisions))
        col3.metric("Transcript Segments", len(diarized))
        st.markdown("---")

        st.subheader("📝 Executive Summary")
        tab1, tab2 = st.tabs([
            "Original (" + LANG_NAMES.get(det_lang, det_lang) + ")",
            "Translated (" + tgt_name + ")"
        ])
        with tab1:
            st.write(summary)
        with tab2:
            st.write(summary_tr)

        st.subheader("🏛️ Key Decisions")
        if decisions:
            for i, d in enumerate(decisions, 1):
                with st.expander("Decision " + str(i) + " — " + d["speaker"] + " (" + d["time"] + ")"):
                    st.write("**Original:** " + d["text"])
                    st.write("**" + tgt_name + ":** " + d["text_tr"])
        else:
            st.info("No decisions extracted.")

        st.subheader("✅ Action Items")
        if actions:
            for i, a in enumerate(actions, 1):
                with st.expander("Action " + str(i) + " — " + a["speaker"] + " (" + a["time"] + ")"):
                    st.write("**Original:** " + a["text"])
                    st.write("**" + tgt_name + ":** " + a["text_tr"])
        else:
            st.info("No action items extracted.")

        st.subheader("🗒️ Full Transcript")
        with st.expander("View full transcript"):
            for s in diarized:
                st.markdown(
                    "**[" + str(s["start"]) + "s] " + s["speaker"] + ":** " +
                    s["text"] + "  \n*🌐 " + s.get("text_tr","") + "*"
                )

        st.markdown("---")
        with st.spinner("📄 Generating DOCX..."):
            docx_path = export_docx(
                summary, summary_tr, actions, decisions,
                diarized, det_lang, target_lang
            )

        with open(docx_path, "rb") as f:
            st.download_button(
                label="📥 Download Meeting Minutes (DOCX)",
                data=f,
                file_name=docx_path,
                mime="application/vnd.openxmlformats-officedocument.wordprocessingml.document"
            )

Overwriting app.py


In [7]:
import subprocess
import time
import threading

# Start Streamlit
proc = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port=8501",
     "--server.headless=true",
     "--server.enableCORS=false"]
)

time.sleep(5)

# Start localtunnel (no account needed)
tunnel = subprocess.Popen(
    ["npx", "localtunnel", "--port", "8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

# Get the URL
for line in iter(tunnel.stdout.readline, b""):
    line = line.decode().strip()
    if "loca.lt" in line:
        print("=" * 50)
        print(f"🌐 App is LIVE at: {line}")
        print("=" * 50)
        print("1. Copy that URL and open it in your browser")
        print("2. If it asks for a password, enter your Colab IP")
        print("   Run this to get your IP:")
        print("   !curl https://ipv4.icanhazip.com")
        break

🌐 App is LIVE at: your url is: https://eight-windows-fetch.loca.lt
1. Copy that URL and open it in your browser
2. If it asks for a password, enter your Colab IP
   Run this to get your IP:
   !curl https://ipv4.icanhazip.com


In [8]:
from google.colab import files
files.download("app.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>